# 1.) Team Records & Standings

- Number of Teams
- Overall Team Record
- Team Home and Away Records
- League Standings
- Divisional Standings

##Teams

In [0]:
%sql
-- Teams
SELECT 
    *
FROM gold.dim_teams
ORDER BY team_id

# Total Record, Home Record, Away Record

In [0]:
%sql
-- Total runs per team
CREATE OR REPLACE TEMP VIEW team_game_runs AS
SELECT
    game_pk,
    team_id,
    SUM(runs) AS runs_scored
FROM gold.fact_batting_stats
GROUP BY game_pk, team_id;


-- Total runs scored per game
CREATE OR REPLACE TEMP VIEW game_results AS
SELECT 
    g.game_pk,
    g.home_team_id,
    h.runs_scored AS home_runs,
    g.away_team_id,
    a.runs_scored AS away_runs
FROM gold.dim_games AS g
LEFT JOIN team_game_runs AS h ON g.game_pk = h.game_pk AND g.home_team_id = h.team_id
LEFT JOIN team_game_runs AS a ON g.game_pk = a.game_pk AND g.away_team_id = a.team_id
WHERE g.game_status IN ('Final', 'Pre-Game', 'Scheduled', 'Completed Early');


-- wins and losses per team per game
CREATE OR REPLACE TEMP VIEW team_outcomes AS
SELECT
    game_pk,
    home_team_id AS team_id,
    CASE WHEN home_runs > away_runs THEN 1 ELSE 0 END AS Win,
    CASE WHEN home_runs < away_runs THEN 1 ELSE 0 END AS Loss,
    home_runs - away_runs AS run_diff
FROM game_results
UNION ALL
SELECT
    game_pk,
    away_team_id AS team_id,
    CASE WHEN away_runs > home_runs THEN 1 ELSE 0 END AS Win,
    CASE WHEN away_runs < home_runs THEN 1 ELSE 0 END AS Loss,
    away_runs - home_runs AS run_diff
FROM game_results;


-- Home wins and losses
CREATE OR REPLACE TEMP VIEW team_home_outcomes AS
SELECT
    home_team_id AS team_id,
    CASE WHEN home_runs > away_runs THEN 1 ELSE 0 END AS Win,
    CASE WHEN home_runs < away_runs THEN 1 ELSE 0 END AS Loss
FROM game_results;


-- Record at Home
CREATE OR REPLACE TEMP VIEW team_home_record AS
SELECT
    t.team_name,
    SUM(Win) AS home_wins,
    SUM(Loss) AS home_losses
FROM team_home_outcomes AS o
LEFT JOIN gold.dim_teams AS t
ON o.team_id = t.team_id
GROUP BY t.team_name;


-- Away wins and losses
CREATE OR REPLACE TEMP VIEW team_away_outcomes AS
SELECT
    away_team_id AS team_id,
    CASE WHEN away_runs > home_runs THEN 1 ELSE 0 END AS Win,
    CASE WHEN away_runs < home_runs THEN 1 ELSE 0 END AS Loss
FROM game_results;


-- Record when Away
CREATE OR REPLACE TEMP VIEW team_away_record AS
SELECT
    t.team_name,
    SUM(Win) AS away_wins,
    SUM(Loss) AS away_losses
FROM team_away_outcomes AS o
LEFT JOIN gold.dim_teams AS t
ON o.team_id = t.team_id
GROUP BY t.team_name;



-- Overall record
CREATE OR REPLACE TEMP VIEW team_record AS
SELECT
    t.team_name,
    SUM(Win) AS Wins,
    SUM(Loss) AS Losses
FROM team_outcomes AS o
LEFT JOIN gold.dim_teams AS t
ON o.team_id = t.team_id
GROUP BY t.team_name;

# Standings

standings

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW standings AS
SELECT 
    t.team_name,
    t.Wins,
    t.Losses,
    h.home_wins,
    h.home_losses,
    a.away_wins,
    a.away_losses 
FROM team_record AS t
LEFT JOIN team_home_record AS h
ON t.team_name = h.team_name
LEFT JOIN team_away_record AS a
ON t.team_name = a.team_name
WHERE t.team_name IS NOT NULL
ORDER BY Wins DESC

In [0]:
%sql

CREATE OR REPLACE TABLE gold.mlb_standings AS

SELECT
    team_name,
    Wins,
    Losses,
    home_wins,
    home_losses,
    away_wins,
    away_losses
FROM standings;

### Manually correcting missing games for certain teams

In [0]:
manual_corrections = [
    {
        "team_name": "Milwaukee Brewers",
        "wins_adjustment": 0,
        "losses_adjustment": 1,
        "home_win_adjustment": 0,
        "home_loss_adjustment": 1,
        "away_win_adjustment": 0,
        "away_loss_adjustment": 0
    },
    {
        "team_name": "Los Angeles Dodgers",
        "wins_adjustment": 0,
        "losses_adjustment": 1,
        "home_win_adjustment": 0,
        "home_loss_adjustment": 0,
        "away_win_adjustment": 0,
        "away_loss_adjustment": 1
    },
    {
        "team_name": "New York Yankees",
        "wins_adjustment": 0,
        "losses_adjustment": 1,
        "home_win_adjustment": 0,
        "home_loss_adjustment": 1,
        "away_win_adjustment": 0,
        "away_loss_adjustment": 0
    },
    {
        "team_name": "Chicago Cubs",
        "wins_adjustment": 2,
        "losses_adjustment": 0,
        "home_win_adjustment": 2,
        "home_loss_adjustment": 0,
        "away_win_adjustment": 0,
        "away_loss_adjustment": 0
    },
    {
        "team_name": "Philadelphia Phillies",
        "wins_adjustment": 1,
        "losses_adjustment": 0,
        "home_win_adjustment": 1,
        "home_loss_adjustment": 0,
        "away_win_adjustment": 0,
        "away_loss_adjustment": 0
    },
    {
        "team_name": "Texas Rangers",
        "wins_adjustment": 0,
        "losses_adjustment": 1,
        "home_win_adjustment": 0,
        "home_loss_adjustment": 1,
        "away_win_adjustment": 0,
        "away_loss_adjustment": 0
    },
    {
        "team_name": "St. Louis Cardinals",
        "wins_adjustment": 1,
        "losses_adjustment": 0,
        "home_win_adjustment": 0,
        "home_loss_adjustment": 0,
        "away_win_adjustment": 1,
        "away_loss_adjustment": 0
    },
    {
        "team_name": "Toronto Blue Jays",
        "wins_adjustment": 0,
        "losses_adjustment": 1,
        "home_win_adjustment": 0,
        "home_loss_adjustment": 0,
        "away_win_adjustment": 0,
        "away_loss_adjustment": 1
    },
    {
        "team_name": "Baltimore Orioles",
        "wins_adjustment": 0,
        "losses_adjustment": 1,
        "home_win_adjustment": 0,
        "home_loss_adjustment": 1,
        "away_win_adjustment": 0,
        "away_loss_adjustment": 0
    },
    {
        "team_name": "Pittsburgh Pirates",
        "wins_adjustment": 1,
        "losses_adjustment": 0,
        "home_win_adjustment": 0,
        "home_loss_adjustment": 0,
        "away_win_adjustment": 1,
        "away_loss_adjustment": 0
    },
    {
        "team_name": "Washington Nationals",
        "wins_adjustment": 0,
        "losses_adjustment": 1,
        "home_win_adjustment": 0,
        "home_loss_adjustment": 0,
        "away_win_adjustment": 0,
        "away_loss_adjustment": 1
    },
    {
        "team_name": "San Francisco Giants",
        "wins_adjustment": 1,
        "losses_adjustment": 0,
        "home_win_adjustment": 0,
        "home_loss_adjustment": 0,
        "away_win_adjustment": 1,
        "away_loss_adjustment": 0
    },
    {
        "team_name": "Los Angeles Angels",
        "wins_adjustment": 1,
        "losses_adjustment": 0,
        "home_win_adjustment": 0,
        "home_loss_adjustment": 0,
        "away_win_adjustment": 1,
        "away_loss_adjustment": 0
    }
]

In [0]:
corrections_df = spark.createDataFrame(manual_corrections)

corrections_df.write.mode("overwrite").saveAsTable("gold.mlb_manual_record_corrections")

corrected_standings

In [0]:
%sql
CREATE OR REPLACE TABLE gold.mlb_standings AS

SELECT

    s.team_name,

    s.Wins
        + COALESCE(c.wins_adjustment, 0)
        AS Wins,

    s.Losses
        + COALESCE(c.losses_adjustment, 0)
        AS Losses,

    s.home_wins
        + COALESCE(c.home_win_adjustment, 0)
        AS home_wins,

    s.home_losses
        + COALESCE(c.home_loss_adjustment, 0)
        AS home_losses,

    s.away_wins
        + COALESCE(c.away_win_adjustment, 0)
        AS away_wins,

    s.away_losses
        + COALESCE(c.away_loss_adjustment, 0)
        AS away_losses

FROM standings s

LEFT JOIN gold.mlb_manual_record_corrections c
    ON s.team_name = c.team_name;

##Checking

In [0]:
%sql

SELECT
    team_name,
    Wins,
    Losses,
    Wins + Losses AS games_played,
    home_wins,
    home_losses,
    away_wins,
    away_losses
FROM gold.mlb_standings
ORDER BY Wins DESC;

team_win_probability

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW team_win_probability AS

SELECT
    team_name,
    Wins,
    Losses,
    home_wins,
    home_losses,
    away_wins,
    away_losses,

    ROUND(
        Wins / (Wins + Losses),
        4
    ) AS win_pct

FROM corrected_standings;

In [0]:
%sql

SELECT *
FROM team_win_probability
ORDER BY win_pct DESC;

# Putting Corrected Standings into Gold Layer so it can be used later

#American League

In [0]:
%sql
SELECT
    l.league_name,
    l.team_name,
    t.Wins,
    t.Losses
FROM gold.dim_teams AS l
LEFT JOIN team_record AS t
ON l.team_name = t.team_name
WHERE league_name = 'American League'
ORDER BY Wins DESC

##American League West

In [0]:
%sql
SELECT
    l.division_name,
    l.team_name,
    t.Wins,
    t.Losses
FROM gold.dim_teams AS l
LEFT JOIN team_record AS t
ON l.team_name = t.team_name
WHERE division_name = 'American League West'
ORDER BY Wins DESC

##American League Central

In [0]:
%sql
SELECT
    l.division_name,
    l.team_name,
    t.Wins,
    t.Losses
FROM gold.dim_teams AS l
LEFT JOIN team_record AS t
ON l.team_name = t.team_name
WHERE division_name = 'American League Central'
ORDER BY Wins DESC

##American League East

In [0]:
%sql
SELECT
    l.division_name,
    l.team_name,
    t.Wins,
    t.Losses
FROM gold.dim_teams AS l
LEFT JOIN team_record AS t
ON l.team_name = t.team_name
WHERE division_name = 'American League East'
ORDER BY Wins DESC

#National League

In [0]:
%sql
SELECT
    l.league_name,
    l.team_name,
    t.Wins,
    t.Losses
FROM gold.dim_teams AS l
LEFT JOIN team_record AS t
ON l.team_name = t.team_name
WHERE league_name = 'National League'
ORDER BY Wins DESC

##National League West

In [0]:
%sql
SELECT
    l.division_name,
    l.team_name,
    t.Wins,
    t.Losses
FROM gold.dim_teams AS l
LEFT JOIN team_record AS t
ON l.team_name = t.team_name
WHERE division_name = 'National League West'
ORDER BY Wins DESC

##National League Central

In [0]:
%sql
SELECT
    l.division_name,
    l.team_name,
    t.Wins,
    t.Losses
FROM gold.dim_teams AS l
LEFT JOIN team_record AS t
ON l.team_name = t.team_name
WHERE division_name = 'National League Central'
ORDER BY Wins DESC

##National League East

In [0]:
%sql
SELECT
    l.division_name,
    l.team_name,
    t.Wins,
    t.Losses
FROM gold.dim_teams AS l
LEFT JOIN team_record AS t
ON l.team_name = t.team_name
WHERE division_name = 'National League East'
ORDER BY Wins DESC

#4.) Run differential vs. Win %

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW run_diff_win_perc AS
SELECT
  t.team_name,
  t.division_name,
  SUM(o.win) AS wins,
  SUM(o.loss) AS losses,
  ROUND((SUM(o.win) / (SUM(o.win) + SUM(o.loss))) * 100, 2) AS win_pct,
  SUM(o.run_diff) AS run_differential
FROM team_outcomes o
JOIN gold.dim_teams t ON o.team_id = t.team_id
GROUP BY t.team_name, t.division_name
ORDER BY win_pct DESC



In [0]:
%sql
WITH runs_scored AS (
  SELECT team_id, SUM(runs) AS total_runs_scored
  FROM gold.fact_batting_stats
  GROUP BY team_id
),
runs_allowed AS (
  SELECT team_id, SUM(runs_allowed) AS total_runs_allowed
  FROM gold.fact_pitching_stats
  GROUP BY team_id
),
pythag AS (
  SELECT
    rs.team_id,
    rs.total_runs_scored,
    ra.total_runs_allowed,
    ROUND(
      POWER(rs.total_runs_scored, 2) * 1.0 /
      NULLIF(POWER(rs.total_runs_scored, 2) + POWER(ra.total_runs_allowed, 2), 0),
    3) AS expected_win_pct,
    ROUND(
        POWER(ra.total_runs_allowed, 2) * 1.0 /
        NULLIF(POWER(rs.total_runs_scored, 2) + POWER(ra.total_runs_allowed, 2), 0),
    3) AS expected_loss_pct
  FROM runs_scored rs
  JOIN runs_allowed ra ON rs.team_id = ra.team_id
)

SELECT
  t.team_name,
  p.expected_win_pct,
  p.expected_loss_pct,
  ROUND(w.wins * 1.0 / (w.wins + w.losses), 3) AS actual_win_pct,
  ROUND((w.wins * 1.0 / (w.wins + w.losses)) - p.expected_win_pct, 3) AS luck_factor,
  ROUND((w.wins * 1.0 / (w.wins + w.losses)) - p.expected_loss_pct, 3) AS anti_luck_factor
FROM pythag p
JOIN gold.dim_teams t ON p.team_id = t.team_id
JOIN standings w ON t.team_name = w.team_name
ORDER BY luck_factor DESC